# Entrenar el ResNet50 Triplet de FORMATEADOR con las vacas de Tres Lomas

Mismo dataset y misma prueba que `entrenar_dinov2.ipynb`, para comparar de igual a igual.

**Antes de empezar**
1. `dataset_dinov2.zip` ya está en tu Drive (lo usa este notebook también).
2. **Si tenés tus pesos de FORMATEADOR** (`resnet50_triplet_best.pth`), subilos a **Mi unidad/vacas/**. Si no los encuentra, arranca desde ResNet50 de ImageNet (funciona igual, solo que la comparación "antes" es contra un ResNet sin entrenar en vacas).
3. **Entorno de ejecución → Cambiar tipo → T4 GPU → Guardar**, y después **Entorno de ejecución → Ejecutar todas**.

Tarda unos 30 a 45 minutos (los recortes sin fondo ya están guardados en Drive desde el entrenamiento de DINOv2).

Resultado en **Mi unidad/vacas/resultado_resnet50/**:
- `resnet50_triplet_vacas.pth`: se carga directo con `EmbeddingExtractor` de `servicioSeguimientoIdentificacion` (misma arquitectura, embedding de 512).
- `metricas.json`: antes / después, y comparación con DINOv2.

In [ ]:
# 1. Drive, GPU y librerías
import os
from google.colab import drive
if os.path.ismount('/content/drive'):
    drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'ultralytics'], check=True)

import torch
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución → Cambiar tipo → T4 GPU, y volvé a ejecutar.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Configuración, dataset y pesos de FORMATEADOR
import json, zipfile, random, time, csv, glob
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from PIL import Image, ImageOps, ImageDraw, ImageFilter

# Cómo se le muestra la vaca a la red:
#   'como_dinov2'      -> recorte sin fondo (SAM), relleno gris hasta cuadrado. Igual que DINOv2:
#                         la comparación es justa. (recomendado)
#   'como_formateador' -> recorte con fondo, estirado a 224x224, como hace hoy
#                         servicioSeguimientoIdentificacion (sin SAM). Usalo si querés enchufarlo
#                         en FORMATEADOR sin agregar SAM al pipeline.
ENTRADA = 'como_dinov2'

MI_UNIDAD = Path('/content/drive/MyDrive')
def buscar(nombre):
    c = sorted(glob.glob(str(MI_UNIDAD / nombre)) + glob.glob(str(MI_UNIDAD / '*' / nombre)))
    return Path(c[0]) if c else None

ZIP = buscar('dataset_dinov2.zip')
assert ZIP, 'No encuentro dataset_dinov2.zip en Mi unidad.'
PESOS_FORMATEADOR = buscar('resnet50_triplet_best.pth')
print('Dataset:', ZIP)
print('Pesos de FORMATEADOR:', PESOS_FORMATEADOR or 'NO encontrados -> se arranca desde ResNet50 de ImageNet')

SALIDA = MI_UNIDAD / 'vacas' / 'resultado_resnet50'
SALIDA.mkdir(parents=True, exist_ok=True)
DIR_DINO = MI_UNIDAD / 'vacas' / 'resultado_dinov2'
TRABAJO = Path('/content/trabajo')

TAM = 224
GRIS = (124, 116, 104)
MEDIA = np.array([0.485, 0.456, 0.406], np.float32)
DESV = np.array([0.229, 0.224, 0.225], np.float32)
SEMILLA = 7
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

if not (TRABAJO / 'dataset_dinov2' / 'meta.csv').exists():
    print('Descomprimiendo dataset...')
    TRABAJO.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(TRABAJO)
DATA = TRABAJO / 'dataset_dinov2'
meta = list(csv.DictReader(open(DATA / 'meta.csv', encoding='utf-8')))
print(Counter(m['split'] for m in meta))

In [ ]:
# 3. Recortes sin fondo (se reutilizan los del entrenamiento de DINOv2; si no están, se calculan con SAM)
MASC = TRABAJO / 'enmascarado'
ZIP_MASC = DIR_DINO / 'enmascarado.zip'
if ZIP_MASC.exists() and not MASC.exists():
    print('Descomprimiendo recortes sin fondo de', ZIP_MASC)
    with zipfile.ZipFile(ZIP_MASC) as z:
        z.extractall(TRABAJO)

def caja_con_margen(box, ancho, alto, margen=0.05):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    return (int(max(0, x1 - w * margen)), int(max(0, y1 - h * margen)),
            int(min(ancho, x2 + w * margen)), int(min(alto, y2 + h * margen)))

def recortar_con_mascara(im, box, poligono, margen=0.05):
    c = caja_con_margen(box, im.width, im.height, margen)
    rec = im.crop(c)
    if poligono is None or len(poligono) < 3:
        return rec
    pts = [(float(x) - c[0], float(y) - c[1]) for x, y in poligono]
    mascara = Image.new('L', rec.size, 0)
    ImageDraw.Draw(mascara).polygon(pts, fill=255)
    mascara = mascara.filter(ImageFilter.MaxFilter(5))
    return Image.composite(rec, Image.new('RGB', rec.size, GRIS), mascara)

def rgba_a_gris(im):
    caja = im.getchannel('A').point(lambda v: 255 if v > 20 else 0).getbbox()
    if caja is None:
        return None
    im = im.crop(caja)
    fondo = Image.new('RGB', im.size, GRIS)
    fondo.paste(im, mask=im.getchannel('A'))
    return fondo

pendientes = [m for m in meta if not (MASC / Path(m['archivo']).with_suffix('.jpg')).exists()]
print(f'Para enmascarar: {len(pendientes)} de {len(meta)}')
if pendientes:
    from ultralytics import SAM
    sam = SAM('sam2.1_t.pt')
    for i, m in enumerate(pendientes, 1):
        destino = MASC / Path(m['archivo']).with_suffix('.jpg')
        destino.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(DATA / m['archivo']) as im:
            im = ImageOps.exif_transpose(im)
            if m['split'] == 'referencia':
                out = (rgba_a_gris(im.convert('RGBA')) if im.mode in ('RGBA', 'LA') else None) or im.convert('RGB')
            else:
                im = im.convert('RGB')
                caja = json.loads(m['caja'])
                poli = None
                try:
                    r = sam.predict(im, bboxes=[caja], device=0, verbose=False)[0]
                    if r.masks is not None and len(r.masks.xy) and len(r.masks.xy[0]) >= 3:
                        poli = r.masks.xy[0].tolist()
                except Exception as e:
                    print('SAM falló en', m['archivo'], e)
                out = recortar_con_mascara(im, caja, poli)
        out.save(destino, quality=92)
        if i % 1000 == 0:
            print(f'  {i}/{len(pendientes)}')

def ruta_imagen(m):
    # 'como_dinov2': recorte sin fondo. 'como_formateador': recorte original con 5% de margen.
    if ENTRADA == 'como_dinov2' or m['split'] == 'referencia':
        return str(MASC / Path(m['archivo']).with_suffix('.jpg'))
    destino = TRABAJO / 'con_fondo' / m['archivo']
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(DATA / m['archivo']) as im:
            im = im.convert('RGB')
            im.crop(caja_con_margen(json.loads(m['caja']), im.width, im.height)).save(destino, quality=92)
    return str(destino)

for m in meta:
    m['img'] = ruta_imagen(m)
print('Imágenes listas. Entrada:', ENTRADA)

In [ ]:
# 4. Limpieza de recorridos largos: exactamente la misma que en el notebook de DINOv2
# (en recorridos de más de 500 frames se saca el 20% de recortes que menos se parecen al resto,
#  medido con el DINOv2 original), así los dos modelos se entrenan y prueban con los mismos datos.
import timm
import torch.nn as nn
import torch.nn.functional as F
DEV = 'cuda'

def tensor_dino(im):
    im = ImageOps.pad(im.convert('RGB'), (TAM, TAM), color=GRIS)
    a = (np.asarray(im, dtype=np.float32) / 255.0 - MEDIA) / DESV
    return torch.from_numpy(a.transpose(2, 0, 1))

@torch.no_grad()
def huellas_dino(modelo, rutas, lote=128):
    modelo.eval(); partes = []
    for i in range(0, len(rutas), lote):
        x = torch.stack([tensor_dino(Image.open(r)) for r in rutas[i:i + lote]]).to(DEV)
        with torch.autocast('cuda', dtype=torch.float16):
            f = modelo(x).float() + modelo(torch.flip(x, dims=[3])).float()
        partes.append(F.normalize(f, dim=1).cpu())
    return torch.cat(partes).numpy()

dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0, img_size=TAM).to(DEV)
largos = defaultdict(list)
for m in meta:
    if m['split'] == 'entrenar' and int(m['largo_recorrido']) > 500:
        largos[m['recorrido']].append(m)
fuera = set()
for rec, ms in largos.items():
    E = huellas_dino(dino, [str(MASC / Path(m['archivo']).with_suffix('.jpg')) for m in ms])
    c = E.mean(0); c /= np.linalg.norm(c)
    sim = E @ c
    corte = np.quantile(sim, 0.2)
    fuera.update(m['archivo'] for m, s in zip(ms, sim) if s < corte)
del dino; torch.cuda.empty_cache()
meta_ok = [m for m in meta if m['archivo'] not in fuera]
print(f'Recorridos largos: {len(largos)}. Recortes descartados: {len(fuera)} (DINOv2 descartó 765)')

In [ ]:
# 5. El modelo de FORMATEADOR (misma arquitectura que servicioSeguimientoIdentificacion/src/embedder.py)
from torchvision import models

class ResNetTriplet(nn.Module):
    # ResNet50 -> Linear(2048->512) -> L2-norm
    def __init__(self, embedding_dim=512, imagenet=True):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if imagenet else None)
        base.fc = nn.Identity()
        self.base = base
        self.fc = nn.Linear(2048, embedding_dim)
    def forward(self, x):
        return F.normalize(self.fc(self.base(x)), p=2, dim=1)

class LegacyEmbeddingNet(nn.Module):
    # checkpoints viejos: ResNet50 -> fc(2048->224) -> BN -> ReLU -> Dropout -> fc(224->dim) -> BN
    def __init__(self, embedding_dim=512):
        super().__init__()
        base = models.resnet50(weights=None)
        base.fc = nn.Sequential(nn.Linear(2048, 224), nn.BatchNorm1d(224), nn.ReLU(inplace=True),
                                nn.Dropout(0.5), nn.Linear(224, embedding_dim), nn.BatchNorm1d(embedding_dim))
        self.base = base
    def forward(self, x):
        return F.normalize(self.base(x), p=2, dim=1)

def nuevo_modelo():
    if PESOS_FORMATEADOR is not None:
        sd = torch.load(PESOS_FORMATEADOR, map_location='cpu')
        if isinstance(sd, dict) and 'state_dict' in sd:
            sd = sd['state_dict']
        m = LegacyEmbeddingNet() if 'base.fc.0.weight' in sd else ResNetTriplet(imagenet=False)
        m.load_state_dict(sd)
        print('Arranca desde tus pesos de FORMATEADOR:', type(m).__name__)
    else:
        m = ResNetTriplet(imagenet=True)
        print('Arranca desde ResNet50 de ImageNet (capa de 512 nueva)')
    return m.to(DEV)

def a_tensor(im):
    if ENTRADA == 'como_dinov2':
        im = ImageOps.pad(im.convert('RGB'), (TAM, TAM), color=GRIS)
    else:  # como FORMATEADOR: transforms.Resize((224, 224)) estira la imagen
        im = im.convert('RGB').resize((TAM, TAM), Image.BILINEAR)
    a = (np.asarray(im, dtype=np.float32) / 255.0 - MEDIA) / DESV
    return torch.from_numpy(a.transpose(2, 0, 1))

@torch.no_grad()
def huellas(modelo, rutas, lote=128):
    modelo.eval(); partes = []
    for i in range(0, len(rutas), lote):
        x = torch.stack([a_tensor(Image.open(r)) for r in rutas[i:i + lote]]).to(DEV)
        with torch.autocast('cuda', dtype=torch.float16):
            f = modelo(x).float() + modelo(torch.flip(x, dims=[3])).float()
        partes.append(F.normalize(f, dim=1).cpu())
    return torch.cat(partes).numpy()

In [ ]:
# 6. Prueba (idéntica a la de DINOv2): recorridos que el modelo nunca vio
def evaluar(modelo, nombre):
    refs = [m for m in meta_ok if m['split'] == 'referencia']
    entr = [m for m in meta_ok if m['split'] == 'entrenar']
    prue = [m for m in meta_ok if m['split'] == 'prueba']
    E_ref = huellas(modelo, [m['img'] for m in refs])
    E_ent = huellas(modelo, [m['img'] for m in entr])
    E_pru = huellas(modelo, [m['img'] for m in prue])
    por_rec = defaultdict(list)
    for m, e in zip(prue, E_pru):
        por_rec[(m['recorrido'], m['vaca'])].append(e)
    Q = np.stack([np.mean(v, 0) for v in por_rec.values()])
    Q /= np.linalg.norm(Q, axis=1, keepdims=True)
    verdad = [v for (_, v) in por_rec]
    res = {}
    for modo, (G, gv) in {
        'A_referencias_y_campo': (np.concatenate([E_ref, E_ent]), [m['vaca'] for m in refs + entr]),
        'B_solo_referencias': (E_ref, [m['vaca'] for m in refs]),
    }.items():
        gv = np.array(gv)
        vacas = sorted(set(gv))
        S = Q @ G.T
        P = np.stack([np.sort(S[:, gv == v], axis=1)[:, -3:].mean(1) for v in vacas], 1)
        orden = np.argsort(-P, axis=1)
        pos = [list(np.array(vacas)[orden[i]]).index(verdad[i]) for i in range(len(verdad))]
        res[modo] = {'top1': float(np.mean([p == 0 for p in pos])),
                     'top5': float(np.mean([p < 5 for p in pos])),
                     'recorridos': len(pos), 'vacas_candidatas': len(vacas)}
    print(f'--- {nombre}')
    for modo, r in res.items():
        print(f"  {modo}: acierto 1°: {r['top1']:.0%}   en top 5: {r['top5']:.0%}   "
              f"({r['recorridos']} recorridos, {r['vacas_candidatas']} vacas candidatas)")
    return res

base = nuevo_modelo()
metricas = {'antes': evaluar(base, 'RESNET50 ANTES (' + ('tus pesos' if PESOS_FORMATEADOR else 'ImageNet') + ')')}
del base; torch.cuda.empty_cache()

In [ ]:
# 7. Entrenamiento
# - Se congelan las primeras capas (conv1, layer1, layer2) y se entrenan layer3, layer4 y la capa de 512.
# - Pérdida ArcFace sobre el embedding de 512 (junta la misma vaca, separa vacas distintas).
# - Mismos aumentos que DINOv2: rotación libre, espejado, luz, blanco y negro a veces.
import torchvision.transforms as T

EPOCAS = 15
LOTE = 64

train = [m for m in meta_ok if m['split'] in ('entrenar', 'referencia')]
vacas = sorted({m['vaca'] for m in train})
idx = {v: i for i, v in enumerate(vacas)}
print(f'Entrenamiento: {len(train)} imágenes de {len(vacas)} vacas')

color = T.Compose([T.RandomApply([T.ColorJitter(0.35, 0.35, 0.25, 0.03)], p=0.8),
                   T.RandomGrayscale(p=0.2),
                   T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2)])

class Datos(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        m = self.items[i]
        im = Image.open(m['img']).convert('RGB')
        ang = random.uniform(-180, 180) if m['split'] != 'referencia' else random.uniform(-15, 15)
        im = im.rotate(ang, resample=Image.BILINEAR, expand=True, fillcolor=GRIS)
        w, h = im.size
        s = random.uniform(0.8, 1.0)
        cw, ch = int(w * s), int(h * s)
        x0, y0 = random.randint(0, w - cw), random.randint(0, h - ch)
        im = im.crop((x0, y0, x0 + cw, y0 + ch))
        if random.random() < 0.5:
            im = ImageOps.mirror(im)
        return a_tensor(color(im)), idx[m['vaca']]

cuenta = Counter(m['vaca'] for m in train)
pesos = [1.0 / np.sqrt(cuenta[m['vaca']]) for m in train]
muestreador = torch.utils.data.WeightedRandomSampler(pesos, num_samples=len(train), replacement=True)
cargador = torch.utils.data.DataLoader(Datos(train), batch_size=LOTE, sampler=muestreador,
                                       num_workers=2, drop_last=True, persistent_workers=True)

class ArcFace(nn.Module):
    def __init__(self, dim, n, s=30.0, m=0.30):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n, dim) * 0.01)
        self.s, self.m = s, m
    def forward(self, f, y):
        cos = F.linear(F.normalize(f), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        objetivo = torch.cos(torch.acos(cos) + self.m)
        uno = F.one_hot(y, cos.shape[1]).bool()
        return F.cross_entropy(self.s * torch.where(uno, objetivo, cos), y)

modelo = nuevo_modelo()
red = modelo.base
congeladas = [red.conv1, red.bn1, red.layer1, red.layer2]
for mod in congeladas:
    for p in mod.parameters():
        p.requires_grad = False
def fijar_bn_congeladas():
    for mod in congeladas:
        mod.eval()  # las BatchNorm congeladas no cambian sus estadísticas

dim = 512
cabeza = ArcFace(dim, len(vacas)).to(DEV)
entrenables = [p for p in modelo.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{'params': entrenables, 'lr': 1e-4},
                         {'params': cabeza.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
pasos = EPOCAS * len(cargador)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-4, 1e-3], total_steps=pasos, pct_start=0.1)
escalador = torch.cuda.amp.GradScaler()

CHECK = SALIDA / f'checkpoint_{ENTRADA}.pt'
inicio = 0
if CHECK.exists():
    ck = torch.load(CHECK, map_location=DEV)
    if ck.get('vacas') == vacas:
        modelo.load_state_dict(ck['modelo']); cabeza.load_state_dict(ck['cabeza'])
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        inicio = ck['epoca'] + 1
        print(f'Retomando desde la época {inicio + 1}')

for ep in range(inicio, EPOCAS):
    modelo.train(); fijar_bn_congeladas()
    t0 = time.time(); total = 0.0
    for x, y in cargador:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            f = modelo(x)
        perdida = cabeza(f.float(), y)
        opt.zero_grad(set_to_none=True)
        escalador.scale(perdida).backward()
        escalador.step(opt); escalador.update(); sched.step()
        total += perdida.item()
    print(f'Época {ep + 1}/{EPOCAS}  pérdida {total / len(cargador):.3f}  ({(time.time() - t0) / 60:.1f} min)')
    torch.save({'modelo': modelo.state_dict(), 'cabeza': cabeza.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'epoca': ep, 'vacas': vacas}, CHECK)

In [ ]:
# 8. Comparar y guardar
metricas['despues'] = evaluar(modelo, 'RESNET50 ENTRENADO')
metricas['config'] = {'entrada': ENTRADA, 'desde': 'pesos FORMATEADOR' if PESOS_FORMATEADOR else 'ImageNet',
                      'epocas': EPOCAS, 'imagenes_entrenamiento': len(train), 'vacas': len(vacas),
                      'descartados_recorridos_largos': len(fuera)}

dino = None
if (DIR_DINO / 'metricas.json').exists():
    dino = json.load(open(DIR_DINO / 'metricas.json'))['despues']
    metricas['dinov2_entrenado'] = dino

print('\nAcierto en 1° lugar (entre los 5 primeros):')
for modo in metricas['antes']:
    a, d = metricas['antes'][modo], metricas['despues'][modo]
    linea = f"  {modo}: ResNet50 {a['top1']:.0%} → {d['top1']:.0%} ({d['top5']:.0%})"
    if dino:
        linea += f"   |   DINOv2 entrenado: {dino[modo]['top1']:.0%} ({dino[modo]['top5']:.0%})"
    print(linea)

# Mismo formato que espera EmbeddingExtractor de servicioSeguimientoIdentificacion
nombre = 'resnet50_triplet_vacas.pth' if ENTRADA == 'como_dinov2' else 'resnet50_triplet_vacas_con_fondo.pth'
torch.save({k: v.float().cpu() for k, v in modelo.state_dict().items()}, SALIDA / nombre)
json.dump(metricas, open(SALIDA / f'metricas_{ENTRADA}.json', 'w'), indent=1, ensure_ascii=False)
print(f'\nGuardado en {SALIDA}: {nombre} y metricas_{ENTRADA}.json')